# GloVe (mean pooling, concatenated) + bigger MLP

**Task:** Abductive Commonsense Reasoning (ART). Given two observations O1, O2 and two
hypotheses H1, H2, pick the hypothesis that better explains what happened in between.

**Approach of this notebook:** Same framework as `w2v_mean_mlp.ipynb`, three changes:

- glove vocabulary (`glove-wiki-gigaword-300`, uncased) -> tokens are lowercased, much lower OOV
- the four sentence vectors are concatenated in fixed order O1, O2, H1, H2 instead of averaged -> 1200 dims, swapping H1 and H2 changes the input
- one more linear layer + ReLU in the MLP

In [1]:
%pip install -q datasets gensim lightning nltk python-dotenv wandb

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from collections import Counter
from pathlib import Path

import gensim.downloader as api
import lightning as L
import nltk
import numpy as np
import torch
import torch.nn as nn
import wandb
from datasets import load_dataset
from dotenv import find_dotenv, load_dotenv
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import WandbLogger
from torch.utils.data import DataLoader, TensorDataset

nltk.download("punkt_tab", quiet=True)

True

In [3]:
dotenv_path = find_dotenv(usecwd=True)
load_dotenv(dotenv_path)
print(f"environment loaded from: {dotenv_path or '<no .env found, using the defaults below>'}")

WANDB_BASE_URL = os.getenv("WANDB_BASE_URL", "https://api.wandb.ai")
WANDB_ENTITY = os.getenv("WANDB_ENTITY", "cmedi32-hochschule-luzern")
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "nlp-project-01-abductive-reasoning")

os.environ["WANDB_BASE_URL"] = WANDB_BASE_URL
os.environ["WANDB_ENTITY"] = WANDB_ENTITY
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

wandb.login()

environment loaded from: /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/.env


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: cmedi32 (cmedi32-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
config = {
    "dataset": "amarfurt/art-nlp-hs26",
    "tokenizer": "nltk.word_tokenize",
    "lowercase": True,
    "remove_punctuation": True,
    "remove_stopwords": False,
    "normalization": "none",
    "truncation": "none",
    "embedding": "glove-wiki-gigaword-300",
    "sentence_representation": "mean",
    "example_representation": "concat",  # O1, O2, H1, H2
    "classifier": "mlp_2_hidden",
    "hidden_dim": 256,
    "learning_rate": 1e-3,
    "batch_size": 64,
    "max_epochs": 100,
    "patience": 5,  # validation checks without improvement of loss/valid before stopping
    "seed": 42,
}

run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name="glove-concatenate-bigger-mlp",
    config=config,
)

In [5]:
dataset = load_dataset("amarfurt/art-nlp-hs26")

train = dataset["train"]
valid = dataset["validation"]
test = dataset["test"]

print(f"train: {len(train)}, valid: {len(valid)}, test: {len(test)}")

train: 16801, valid: 1000, test: 1532


In [6]:
def preprocess(text):
    tokens = nltk.word_tokenize(text)
    return [token.lower() for token in tokens if any(character.isalnum() for character in token)]

In [7]:
TEXT_FIELDS = ["observation_1", "observation_2", "hypothesis_1", "hypothesis_2"]

def preprocess_example(example):
    processed = {field: preprocess(example[field]) for field in TEXT_FIELDS}
    processed["target"] = example["label"] - 1
    return processed


def preprocess_split(split):
    return split.map(preprocess_example)

In [8]:
train_tokens = preprocess_split(train)
valid_tokens = preprocess_split(valid)
test_tokens = preprocess_split(test)

Map:   0%|          | 0/16801 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1532 [00:00<?, ? examples/s]

In [9]:
wv = api.load(config["embedding"])
print(f"vocabulary: {len(wv.index_to_key)} words, {wv.vector_size} dimensions")

vocabulary: 400000 words, 300 dimensions


In [10]:
oov_counts = Counter()
embed_stats = Counter()

def embed(tokens):
    vectors = []
    for token in tokens:
        embed_stats["tokens"] += 1
        if token in wv:
            vectors.append(wv[token])
        else:
            embed_stats["oov"] += 1
            oov_counts[token] += 1
    if not vectors:
        embed_stats["empty_sentences"] += 1
        return np.zeros(wv.vector_size)
    return np.mean(vectors, axis=0)

In [11]:
def embed_example(example):
    vectors = [embed(example[field]) for field in TEXT_FIELDS]
    return np.concatenate(vectors) # this simple change in combination with much lower OOV rate made a huge difference.

In [12]:
X_train = np.stack([embed_example(example) for example in train_tokens])
y_train = np.array(train_tokens["target"])
X_valid = np.stack([embed_example(example) for example in valid_tokens])
y_valid = np.array(valid_tokens["target"])
X_test = np.stack([embed_example(example) for example in test_tokens])
y_test = np.array(test_tokens["target"])

### TFIDF Vectorizer

### DataLoaders - Takes care of Training Batching and Loading

In [13]:
L.seed_everything(config["seed"])

def make_loader(X, y, shuffle):
    dataset = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    return DataLoader(dataset, batch_size=config["batch_size"], shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
valid_loader = make_loader(X_valid, y_valid, shuffle=False)
test_loader = make_loader(X_test, y_test, shuffle=False)

Seed set to 42


In [ ]:
### LightningModule Mode

### Best Checkpoint + Upload Artifacts to Weights and Biases